# 🖼️ Image Compressor

Compress multiple images using **Pillow** in Google Colab.

### Features
- Upload multiple images at once
- Compress JPG, JPEG, PNG, and WEBP images
- Control JPEG quality
- Preserve image dimensions
- Automatically create a ZIP file
- Automatically download the ZIP file


## 1. Install Dependencies

In [ ]:
!pip install -q Pillow

## 2. Upload Images

Select multiple images from your computer.

In [ ]:
from google.colab import files

uploaded = files.upload()

image_files = [
    filename
    for filename in uploaded.keys()
    if filename.lower().endswith(
        ('.jpg', '.jpeg', '.png', '.webp')
    )
]

print(f"Uploaded {len(image_files)} image(s):")

for image_file in image_files:
    print(f"  - {image_file}")

## 3. Set Compression Quality

For JPEG/WebP images, a quality of around **75–85** is usually a good balance between file size and image quality.

- `95` = very high quality / larger file
- `85` = high quality
- `75` = good compression
- `60` = stronger compression
- `40` = low quality / small file


In [ ]:
QUALITY = 80

print(f"Compression quality: {QUALITY}")

## 4. Compress Images

In [ ]:
from PIL import Image
import os

output_folder = 'compressed_images'
os.makedirs(output_folder, exist_ok=True)

compressed_files = []

for image_file in image_files:
    try:
        input_path = image_file

        name, extension = os.path.splitext(image_file)
        extension_lower = extension.lower()

        output_file = os.path.join(
            output_folder,
            f"{name}_compressed{extension_lower}"
        )

        image = Image.open(input_path)

        original_size = os.path.getsize(input_path)

        if extension_lower in ('.jpg', '.jpeg'):
            if image.mode in ('RGBA', 'LA', 'P'):
                image = image.convert('RGB')

            image.save(
                output_file,
                'JPEG',
                quality=QUALITY,
                optimize=True
            )

        elif extension_lower == '.webp':
            image.save(
                output_file,
                'WEBP',
                quality=QUALITY,
                method=6
            )

        elif extension_lower == '.png':
            image.save(
                output_file,
                'PNG',
                optimize=True,
                compress_level=9
            )

        compressed_size = os.path.getsize(output_file)

        reduction = (1 - compressed_size / original_size) * 100

        compressed_files.append(output_file)

        print(f"✓ {image_file}")
        print(f"  Original:  {original_size / 1024:.1f} KB")
        print(f"  Compressed: {compressed_size / 1024:.1f} KB")
        print(f"  Reduction: {reduction:.1f}%\n")

    except Exception as e:
        print(f"✗ Failed: {image_file}")
        print(f"  Error: {e}\n")

print(f"Compressed {len(compressed_files)} image(s).")

## 5. Create ZIP File and Download Automatically

In [ ]:
import zipfile
from google.colab import files

zip_name = 'compressed_images.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for compressed_file in compressed_files:
        zipf.write(
            compressed_file,
            arcname=os.path.basename(compressed_file)
        )

print(f"✓ Created: {zip_name}")
print(f"✓ Contains {len(compressed_files)} image(s)")

# Automatically download the ZIP
files.download(zip_name)

## Output

For example:

```text
photo1.jpg
photo2.png
photo3.webp
```

becomes:

```text
photo1_compressed.jpg
photo2_compressed.png
photo3_compressed.webp
```

The notebook then creates and downloads:

```text
compressed_images.zip
```